In [0]:
#### Loading libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Question 13

**Level:**
Intermediate

**Question:**
You are given an employee dataset. Write a PySpark query to return **only the highest-paid employee from each department**.
If there is a tie, return just one of them. Your final output must retain all original columns (`emp_id`, `name`, `department`, `salary`).

**Practical Use Case:**
This is one of the most common Data Engineering interview questions and real-world tasks. It is frequently used for **deduplication by recency** (e.g., "get the latest transaction for each user") or intra-group ranking. You cannot solve this purely with a `groupBy().max()` because grouping by department will drop the `emp_id` and `name` columns.

**PySpark Query for Table Creation:**

```python
data = [
    (1, "Alice", "Sales", 60000),
    (2, "Bob", "Engineering", 120000),
    (3, "Charlie", "Sales", 75000),
    (4, "Diana", "Engineering", 120000), # Tied for highest in Engineering
    (5, "Eve", "HR", 50000),
    (6, "Frank", "Engineering", 105000)
]

columns = ["emp_id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**
*(Note: Because Bob and Diana tied for Engineering, returning either one of them is acceptable. Below shows Bob being selected, but Diana is equally correct).*

| emp_id | name | department | salary |
| --- | --- | --- | --- |
| 2 | Bob | Engineering | 120000 |
| 5 | Eve | HR | 50000 |
| 3 | Charlie | Sales | 75000 |

*Waiting for your PySpark solution.*

In [0]:
## Question: 
data = [
    (1, "Alice", "Sales", 60000),
    (2, "Bob", "Engineering", 120000),
    (3, "Charlie", "Sales", 75000),
    (4, "Diana", "Engineering", 120000), # Tied for highest in Engineering
    (5, "Eve", "HR", 50000),
    (6, "Frank", "Engineering", 105000)
]

columns = ["emp_id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer
df.withColumn('rn', row_number().over(Window.partitionBy('department').orderBy(col('salary').desc())))\
    .filter(col('rn') == 1)\
        .select('emp_id', 'name', 'department', 'salary')\
            .display()

### Question 14

**Level:**
Intermediate

**Question:**
You are ingesting a JSON payload from an e-commerce API. The API returns an array of `purchased_items` for each user.
Write a PySpark query to "flatten" this data so that **each item has its own row**.

* If a user has an empty array `[]` or a `NULL` value for `purchased_items`, they should be completely **removed** from the final output.

**Practical Use Case:**
Dealing with nested arrays and structs (like JSON data) is a massive part of modern ETL. Downstream relational databases (like Snowflake or Redshift) prefer tabular, flattened data. The process of converting an array column into individual rows is required to join this data with a standard `products` dimension table.

**PySpark Query for Table Creation:**

```python
data = [
    (1, "Alice", ["Laptop", "Mouse"]),
    (2, "Bob", ["Keyboard"]),
    (3, "Charlie", []),        # Empty array (no purchases)
    (4, "Diana", None)         # Null (no purchases)
]

columns = ["user_id", "name", "purchased_items"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| user_id | name | item |
| --- | --- | --- |
| 1 | Alice | Laptop |
| 1 | Alice | Mouse |
| 2 | Bob | Keyboard |

*(Notice that the `purchased_items` column is renamed to `item`, and Charlie/Diana are dropped).*

*Waiting for your PySpark solution.*

In [0]:
## Question:
data = [
    (1, "Alice", ["Laptop", "Mouse"]),
    (2, "Bob", ["Keyboard"]),
    (3, "Charlie", []),        # Empty array (no purchases)
    (4, "Diana", None)         # Null (no purchases)
]

columns = ["user_id", "name", "purchased_items"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Solution:
df.withColumn('item', explode(col('purchased_items')))\
    .select('user_id', 'name', 'item')\
        .display()

### Question 15

**Level:**
Intermediate

**Question:**
Now let's do the exact reverse. You are given a flattened dataset tracking the web pages visited by users. Notice that Alice visited the "Home" page twice.

Write a PySpark query to group the data so that there is only **one row per user**. You need to create a new column called `unique_pages` that contains an **Array (list)** of the pages they visited.
*Critically, the array must NOT contain any duplicate pages.*

**Practical Use Case:**
This is the "Roll-up" phase of ETL. Data Engineers frequently take massive, flattened event logs (like clickstreams or transaction ledgers) and aggregate them back up into user-level profile tables. Storing a list of unique interactions as an Array inside a single row is highly optimized for downstream Machine Learning models or NoSQL databases.

**PySpark Query for Table Creation:**

```python
data = [
    (1, "Alice", "Home"),
    (1, "Alice", "Checkout"),
    (1, "Alice", "Home"),      # Duplicate visit
    (2, "Bob", "Home"),
    (3, "Charlie", "Contact")
]

columns = ["user_id", "name", "page_visited"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**
*(Note: Row order and the order of elements inside the arrays may vary due to distributed processing).*

| user_id | name | unique_pages |
| --- | --- | --- |
| 1 | Alice | [Home, Checkout] |
| 2 | Bob | [Home] |
| 3 | Charlie | [Contact] |

*Waiting for your PySpark solution.*

In [0]:
## Question:
data = [
    (1, "Alice", "Home"),
    (1, "Alice", "Checkout"),
    (1, "Alice", "Home"),      # Duplicate visit
    (2, "Bob", "Home"),
    (3, "Charlie", "Contact")
]

columns = ["user_id", "name", "page_visited"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Solution:
df.groupBy('user_id', 'name').agg(collect_set('page_visited').alias('unique_pages')).display()

### Question 16

**Level:**
Intermediate

**Question:**
You are processing transaction data for a BI dashboard. The reporting team wants to see the total sales per user, but they want the `category` values to be transformed into **columns**.
Write a PySpark query to reshape the data so that:

1. Each `user_id` has exactly one row.
2. There is a column for total **Electronics** sales and a column for total **Clothing** sales.
3. The values in those columns should be the `sum` of the `amount` for that user/category.

**Practical Use Case:**
This is called **Pivoting**. It is incredibly common when preparing data for Machine Learning (creating user-feature matrices) or preparing cross-tab reports for financial analysts. We are moving data from a "long" format to a "wide" format.

**PySpark Query for Table Creation:**

```python
data = [
    (1, "Electronics", 200.00),
    (1, "Clothing", 50.00),
    (1, "Electronics", 100.00), # Alice bought two electronics
    (2, "Clothing", 75.00),     # Bob only bought clothing
    (3, "Electronics", 300.00)  # Charlie only bought electronics
]

columns = ["user_id", "category", "amount"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| user_id | Clothing | Electronics |
| --- | --- | --- |
| 1 | 50.0 | 300.0 |
| 2 | 75.0 | null |
| 3 | null | 300.0 |

*(Note: NULLs are expected if a user did not buy anything in that category).*

*Waiting for your PySpark solution.*

In [0]:
## Question:
data = [
    (1, "Electronics", 200.00),
    (1, "Clothing", 50.00),
    (1, "Electronics", 100.00), # Alice bought two electronics
    (2, "Clothing", 75.00),     # Bob only bought clothing
    (3, "Electronics", 300.00)  # Charlie only bought electronics
]

columns = ["user_id", "category", "amount"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Solution
df.groupBy('user_id').pivot('category').agg(sum('amount')).display()

### Question 17

**Level:**
Intermediate

**Question:**
You are analyzing user login behavior to build a retention metric. You need to find out when each user *previously* logged in.

Write a PySpark query to create a new column called `previous_login_date`.

* This column should contain the user's `login_date` from the **row immediately preceding** their current login.
* If it is the user's very first login, the `previous_login_date` should be `null`.

**Practical Use Case:**
Calculating time-between-events (e.g., days between purchases, gap between app sessions) is a foundational step in behavioral analytics and event-driven ETL. To do this, you must compare a row to its preceding row, which requires advanced Window functions.

**PySpark Query for Table Creation:**

```python
from pyspark.sql.functions import to_date

data = [
    (1, "2023-10-01"), # User 1's 1st login
    (1, "2023-10-05"), # User 1's 2nd login
    (1, "2023-10-12"), # User 1's 3rd login
    (2, "2023-11-01"), # User 2's 1st login
    (2, "2023-11-03")  # User 2's 2nd login
]

columns = ["user_id", "login_date_str"]

df = spark.createDataFrame(data, columns).withColumn("login_date", to_date("login_date_str")).drop("login_date_str")
df.display()

```

**Expected Output:**

| user_id | login_date | previous_login_date |
| --- | --- | --- |
| 1 | 2023-10-01 | null |
| 1 | 2023-10-05 | 2023-10-01 |
| 1 | 2023-10-12 | 2023-10-05 |
| 2 | 2023-11-01 | null |
| 2 | 2023-11-03 | 2023-11-01 |

*Waiting for your PySpark solution.*

In [0]:
data = [
    (101, "2026-09-01"),
    (101, "2026-09-02"),
    (101, "2026-09-03"),
    (101, "2026-09-05"),
    (101, "2026-09-08"),
    
    (102, "2026-09-01"),
    (102, "2026-09-03"),
    (102, "2026-09-04"),
    (102, "2026-09-05"),
    (102, "2026-09-07"),
    
    (103, "2026-09-02"),
    (103, "2026-09-03"),
    (103, "2026-09-06"),
    (103, "2026-09-07"),
    (103, "2026-09-08"),
]

columns = ["user_id", "login_date"]

df = spark.createDataFrame(data, columns)

df = df.withColumn(
    "login_date",
    to_date("login_date")
)

df.orderBy("user_id", "login_date").display()

In [0]:
df.createOrReplaceTempView('df')

In [0]:
spark.sql('''
          with cte as (
          select *, row_number() over(partition by user_id order by login_date) as rn
          from df
          )
          select *, login_date - rn as diff from cte
          
          ''').display()